# Hotel Customer No-Show Prediction — Exploratory Data Analysis

## Objective and evaluation framing

The target is `no_show`: `0` means the guest arrived and `1` means no-show. This is binary classification. A missed no-show can leave inventory unsold, while a false alert can create unnecessary customer intervention. The modelling stage therefore compares imbalance strategies using **PR-AUC**, checks probability calibration, and chooses a decision threshold on validation data using an explicit false-negative:false-positive cost ratio. The test set remains untouched until final evaluation.

## 1. Setup and simple CSV loading

In [ ]:
from pathlib import Path
import calendar
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from scipy.stats import chi2_contingency
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

DATA_PATH = Path.cwd() / "data" / "data.csv"
df_raw = pd.read_csv(DATA_PATH)
print(f"Dataset loaded: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
display(df_raw.head())

## 2. Sampling decision

The full dataset is retained for data-quality checks, descriptive summaries, statistical tests, and final modelling because 119,391 rows fit comfortably in memory. For repeated exploratory charts and the quick feature-importance screen, a reproducible 30,000-row stratified sample is used to reduce rendering and fitting time while preserving the no-show rate. This is an analysis convenience, not a reduction of the training data.

In [ ]:
valid_target = pd.to_numeric(df_raw["no_show"], errors="coerce")
sampling_source = df_raw.loc[valid_target.isin([0, 1])].copy()
eda_sample, _ = train_test_split(
    sampling_source,
    train_size=30_000,
    random_state=42,
    stratify=sampling_source["no_show"],
)
sampling_check = pd.DataFrame({
    "rows": [len(sampling_source), len(eda_sample)],
    "no_show_rate": [sampling_source["no_show"].mean(), eda_sample["no_show"].mean()],
}, index=["Full usable data", "EDA sample"])
display(sampling_check)

## 3. Schema and data-quality audit

In [ ]:
required_columns = {
    "booking_id", "no_show", "branch", "booking_month", "arrival_month",
    "arrival_day", "checkout_month", "checkout_day", "country", "first_time",
    "room", "price", "platform", "num_adults", "num_children",
}
target_numeric = pd.to_numeric(df_raw["no_show"], errors="coerce")
audit = pd.DataFrame({
    "check": ["Required schema", "Shape", "Duplicate rows", "Missing cells", "Invalid target rows", "Unique booking IDs", "Negative checkout days"],
    "result": [
        "Pass" if required_columns.issubset(df_raw.columns) else "Fail",
        f"{df_raw.shape[0]:,} x {df_raw.shape[1]}",
        int(df_raw.duplicated().sum()),
        int(df_raw.isna().sum().sum()),
        int((~target_numeric.isin([0, 1])).sum()),
        int(df_raw["booking_id"].nunique()),
        int((df_raw["checkout_day"] < 0).sum()),
    ],
})
display(audit)
column_audit = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "non_null": df_raw.notna().sum(),
    "missing": df_raw.isna().sum(),
    "missing_percent": df_raw.isna().mean() * 100,
    "unique": df_raw.nunique(dropna=True),
}).sort_values("missing_percent", ascending=False)
display(column_audit)

## 4. Missingness structure and target relationship

In [ ]:
missing = column_audit.loc[column_audit["missing"] > 0, ["missing", "missing_percent"]]
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.barplot(data=missing.reset_index(), x="missing_percent", y="index", color="#4C78A8", ax=axes[0])
axes[0].set(title="Missingness by feature", xlabel="Missing values (%)", ylabel="Feature")
missing_matrix = df_raw[["room", "price"]].isna().sample(min(1000, len(df_raw)), random_state=42)
sns.heatmap(missing_matrix.T, cbar=False, yticklabels=True, xticklabels=False, cmap="Blues", ax=axes[1])
axes[1].set_title("Room and price missingness (1,000-row sample)")
plt.tight_layout(); plt.show()

missing_target = []
for column in ["room", "price"]:
    summary = df_raw.assign(is_missing=df_raw[column].isna()).groupby("is_missing")["no_show"].agg(["count", "mean"])
    summary["feature"] = column
    missing_target.append(summary.reset_index())
display(pd.concat(missing_target, ignore_index=True).rename(columns={"mean": "no_show_rate"}))

`room` and `price` have substantial missingness. Missingness is retained through pipeline indicators instead of being silently erased. A mode/median value is learned inside each training fold only, preventing validation or test information from entering preprocessing.

## 5. Transparent cleaning and feature engineering for analysis

In [ ]:
df = df_raw.loc[target_numeric.isin([0, 1])].copy()
df["no_show"] = df["no_show"].astype(int)
text_columns = ["branch", "booking_month", "arrival_month", "checkout_month", "country", "first_time", "room", "platform"]
for column in text_columns:
    df[column] = df[column].astype(object).str.strip().str.title()
adult_text = df["num_adults"].astype("string").str.strip().str.lower()
df["num_adults"] = pd.to_numeric(adult_text.replace({"one": "1", "two": "2"}), errors="coerce")
df["num_children"] = pd.to_numeric(df["num_children"], errors="coerce")
df["checkout_day"] = pd.to_numeric(df["checkout_day"], errors="coerce").abs()
df["price_currency"] = df["price"].str.extract(r"^([A-Za-z]{3})\$")[0].str.upper()
df["price_amount"] = pd.to_numeric(df["price"].str.replace(r"^[A-Za-z]{3}\$\s*", "", regex=True), errors="coerce")
df["price_sgd"] = df["price_amount"] * df["price_currency"].map({"SGD": 1.0, "USD": 1.35})

month_number = {month: index for index, month in enumerate(calendar.month_name) if month}
month_start = dict(zip(calendar.month_name[1:], np.cumsum([0, 31, 29, 31, 30, 31, 30, 31, 31, 30, 31, 30])))
booking_number = df["booking_month"].map(month_number)
arrival_number = df["arrival_month"].map(month_number)
df["booking_lead_months"] = (arrival_number - booking_number) % 12
arrival_ordinal = df["arrival_month"].map(month_start) + df["arrival_day"]
checkout_ordinal = df["checkout_month"].map(month_start) + df["checkout_day"]
df["length_of_stay_days"] = (checkout_ordinal - arrival_ordinal) % 366
df.loc[df["length_of_stay_days"] > 31, "length_of_stay_days"] = np.nan
df["total_guests"] = df["num_adults"] + df["num_children"]
df["price_per_guest_sgd"] = df["price_sgd"] / df["total_guests"].replace(0, np.nan)
df["branch_country"] = df["branch"].fillna("Missing") + "__" + df["country"].fillna("Missing")
df["branch_first_time"] = df["branch"].fillna("Missing") + "__" + df["first_time"].fillna("Missing")
df["country_first_time"] = df["country"].fillna("Missing") + "__" + df["first_time"].fillna("Missing")
print(f"Prepared EDA data: {df.shape[0]:,} rows x {df.shape[1]} columns")

## 6. Descriptive statistics and class imbalance

In [ ]:
display(df[["no_show", "arrival_day", "checkout_day", "price_sgd", "num_adults", "num_children", "booking_lead_months", "length_of_stay_days", "total_guests"]].describe().T)
target_summary = df["no_show"].value_counts().sort_index().to_frame("count")
target_summary["proportion"] = df["no_show"].value_counts(normalize=True).sort_index()
display(target_summary)
imbalance_ratio = target_summary.loc[0, "count"] / target_summary.loc[1, "count"]
print(f"Majority:minority ratio = {imbalance_ratio:.2f}:1")
sns.countplot(data=df, x="no_show", color="#4C78A8")
plt.xticks([0, 1], ["Show", "No-show"]); plt.title("Target distribution"); plt.show()

No-shows are 37.0% of usable bookings, a moderate 1.70:1 imbalance. Accuracy is therefore insufficient on its own. The modelling plan explicitly compares an unweighted model, class weighting, random oversampling performed only within each training fold, and a dummy prior baseline. PR-AUC is the selection metric because it focuses on minority-class ranking. Recall, precision, F2, balanced accuracy, MCC, ROC-AUC, calibration, and confusion counts are also reported.

## 7. Temporal analysis

In [ ]:
month_order = list(calendar.month_name[1:])
arrival_rates = df.groupby("arrival_month")["no_show"].agg(["count", "mean"]).reindex(month_order)
booking_rates = df.groupby("booking_month")["no_show"].agg(["count", "mean"]).reindex(month_order)
fig, axes = plt.subplots(1, 2, figsize=(17, 5))
sns.lineplot(x=month_order, y=arrival_rates["mean"], marker="o", ax=axes[0])
sns.lineplot(x=month_order, y=booking_rates["mean"], marker="o", ax=axes[1])
for axis, title in zip(axes, ["No-show rate by arrival month", "No-show rate by booking month"]):
    axis.set_title(title); axis.set_ylabel("No-show rate"); axis.tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.show()

lead_summary = df.groupby("booking_lead_months")["no_show"].agg(["count", "mean"])
display(lead_summary.rename(columns={"mean": "no_show_rate"}))
sns.barplot(data=lead_summary.reset_index(), x="booking_lead_months", y="mean", color="#F28E2B")
plt.ylabel("No-show rate"); plt.title("No-show rate by approximate booking lead months"); plt.show()

Months are ordered chronologically rather than alphabetically. The pipeline also creates sine/cosine month encodings so December and January remain adjacent. Lead time is approximate because the dataset contains no year; modulo-12 differences are useful but cannot distinguish bookings made more than one year ahead.

## 8. Subgroups and implemented interaction candidates

In [ ]:
rate_features = ["branch", "country", "first_time", "room", "platform", "arrival_month"]
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
for column, axis in zip(rate_features, axes.flatten()):
    rates = df.groupby(column, dropna=False)["no_show"].mean().sort_values(ascending=False)
    sns.barplot(x=rates.index.astype(str), y=rates.values, ax=axis, color="#E15759")
    axis.set_title(f"No-show rate by {column}"); axis.set_ylabel("No-show rate"); axis.tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.show()

display(pd.pivot_table(df, values="no_show", index="country", columns=["branch", "first_time"], aggfunc=["mean", "count"]))
interaction_rates = df.groupby("country_first_time")["no_show"].agg(["count", "mean"]).query("count >= 100").sort_values("mean", ascending=False)
display(interaction_rates.rename(columns={"mean": "no_show_rate"}))

The subgroup patterns motivate explicit `branch_country`, `branch_first_time`, and `country_first_time` interaction features. These are tested downstream rather than assumed to be causal. Country-related outputs require fairness monitoring and must not be used as an automatic basis for rejecting bookings.

## 9. Statistical checks and effect sizes

In [ ]:
categorical_tests = []
for column in rate_features + ["branch_country", "branch_first_time", "country_first_time"]:
    table = pd.crosstab(df[column].fillna("Missing"), df["no_show"])
    chi2, p_value, _, expected = chi2_contingency(table)
    cramer_v = np.sqrt(chi2 / (table.values.sum() * (min(table.shape) - 1)))
    categorical_tests.append({
        "feature": column,
        "minimum_group_size": int(table.sum(axis=1).min()),
        "minimum_expected_count": float(expected.min()),
        "p_value": p_value,
        "cramers_v": cramer_v,
    })
categorical_tests = pd.DataFrame(categorical_tests).sort_values("cramers_v", ascending=False)
display(categorical_tests.style.format({"p_value": "{:.2e}", "cramers_v": "{:.3f}", "minimum_expected_count": "{:.1f}"}))

The chi-square checks use independent booking rows and mutually exclusive categories. Group sizes and minimum expected counts are shown to diagnose sparse-table risk rather than reporting p-values alone. With this large dataset, small p-values are expected; Cramér’s V is more useful for judging practical strength.

## 10. Exploratory model-based feature importance

In [ ]:
importance_columns = [
    "branch", "country", "first_time", "room", "platform", "arrival_month",
    "branch_country", "branch_first_time", "country_first_time", "arrival_day",
    "checkout_day", "price_sgd", "num_adults", "num_children",
    "booking_lead_months", "length_of_stay_days", "total_guests", "price_per_guest_sgd",
]
model_sample, _ = train_test_split(df, train_size=30_000, random_state=42, stratify=df["no_show"])
X_importance = pd.get_dummies(model_sample[importance_columns], dummy_na=True)
X_importance = X_importance.fillna(X_importance.median(numeric_only=True)).fillna(0)
y_importance = model_sample["no_show"]
screening_model = RandomForestClassifier(
    n_estimators=120, max_depth=12, min_samples_leaf=10,
    class_weight="balanced_subsample", random_state=42, n_jobs=1,
)
screening_model.fit(X_importance, y_importance)
importance = pd.Series(screening_model.feature_importances_, index=X_importance.columns).sort_values(ascending=False).head(20)
display(importance.to_frame("importance"))
importance.sort_values().plot.barh(figsize=(9, 7), color="#59A14F")
plt.title("Exploratory Random Forest importance (stratified sample)"); plt.tight_layout(); plt.show()

This is an exploratory screen, not final model evaluation. It checks whether the EDA-derived temporal and interaction features carry predictive signal. Final permutation importance is calculated on a separate validation partition after model selection and is saved with the training artifacts.

## 11. EDA-to-modelling decisions

In [ ]:
decisions = pd.DataFrame([
    ["Moderate 37% minority class", "Compare unweighted, class-weighted and in-fold oversampling strategies; include dummy baseline"],
    ["Business costs are asymmetric", "Choose threshold on validation data with configurable 5:1 FN:FP cost and precision guardrail"],
    ["Probability quality matters", "Calibrate the selected model and report Brier score, log loss and calibration curve"],
    ["Country and branch are strong subgroup signals", "Test explicit branch-country and first-time interactions; document fairness risk"],
    ["Month and timing patterns are ordered/cyclical", "Add lead months, stay length, and sine/cosine month features"],
    ["Room and price have material missingness", "Fold-local imputation plus missingness indicators"],
    ["booking_id is unique", "Exclude identifier and assert target/ID absence from model inputs"],
    ["Model comparison must be threshold-independent", "Select by five-fold PR-AUC, then tune threshold separately"],
], columns=["EDA finding", "Modelling decision"])
display(decisions)

## 12. Known limitations

- The 5:1 false-negative:false-positive cost ratio is a transparent scenario, not an audited monetary estimate. Hotel finance and operations should replace it with actual intervention and empty-room costs.
- Month fields have no year, so lead time and stay length are approximate. A time-based holdout cannot be constructed.
- The fixed USD-to-SGD rate (1.35) is an assumption because transaction dates are unavailable.
- Country is strongly associated with historical outcomes but may encode structural or demographic differences. It requires fairness review, subgroup monitoring, and human-governed policy use.
- No free-text field exists, so text-derived analysis is not applicable.
- Feature importance is predictive, not causal.
- Performance must be revalidated on newer bookings and monitored for calibration and drift before deployment.

The analysis supports a leakage-safe, cost-aware classification workflow but not automatic booking rejection.